# Language-model surfaces on Kaggle (7B models)

**Before running** (right-hand panel, *Session options*):
1. *Accelerator*: **GPU T4 x2** (two 16 GB cards).
2. *Internet*: **on** (needs a phone-verified Kaggle account).
3. *Input*: **+ Add Input → Datasets → your uploaded `gpu_bundle`** (upload
   `gpu_bundle.zip` once as a private dataset; Kaggle unpacks it).

**Run**: *Save Version → Save & Run All (Commit)*. It runs in the background for
up to 12 hours, even with the browser closed. The script stops itself about 30
minutes before the limit and packs everything.

**Afterwards**: open the saved version → *Output* → download
`out/gpu_results.zip`. That file is all that needs to come back.

**To continue** in a new session (if the log says *paused (time limit)*):
*+ Add Input → Your Work → this notebook* (its last version's output), then
*Save & Run All* again. Finished settings are never redone.

Set `DRY_RUN = True` for a 3-minute test of the whole pipeline first.

In [ ]:
DRY_RUN = False          # True: quick test with a 135M model
MODELS = "mistral7b,olmo7b,qwen7b"
TIME_LIMIT_HOURS = 11.5  # Kaggle sessions stop at 12 h

In [ ]:
import glob, os, shutil, subprocess
runner = sorted(glob.glob("/kaggle/input/**/run_gpu.py", recursive=True))
assert runner, "gpu_bundle not found: add your uploaded gpu_bundle dataset as an input"
bundle = os.path.dirname(runner[0])
work = "/kaggle/working/out"
cands = [d for d in ["/kaggle/temp", "/kaggle/tmp", "/tmp"] if os.path.isdir(d)]
cache_root = max(cands, key=lambda d: shutil.disk_usage(d).free)
cache = os.path.join(cache_root, "hf_cache")
print("bundle:", bundle)
print("model cache:", cache, f"({shutil.disk_usage(cache_root).free / 1e9:.0f} GB free)")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r {bundle}/requirements-gpu.txt

In [ ]:
args = f"--out {work} --hf-cache {cache} --time-limit {TIME_LIMIT_HOURS} --margin 30 --delete-models-after --resume-from /kaggle/input"
args += " --dry-run" if DRY_RUN else f" --models {MODELS}"
!cd /kaggle/working && python {bundle}/run_gpu.py {args}

In [ ]:
!ls -la /kaggle/working/out && cat /kaggle/working/out/run_info.json | head -60